In [ ]:
# Ejecuta esta celda. Balisco preparó la cola y la ubicación de tu biblioteca.
ROOT_PARTS = __BALISCO_ROOT__
import io, json, os, re, subprocess, sys, zipfile
from pathlib import Path
from PIL import Image, ImageOps, ImageDraw
from google.colab import drive
drive.mount('/content/drive')
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'pypdfium2>=5,<6'], check=True)
import pypdfium2 as pdfium
root = Path('/content/drive/MyDrive').joinpath(*ROOT_PARTS)
hidden = root / '.balisco'
queue = json.loads((hidden / 'cola_render.json').read_text(encoding='utf-8'))
assert queue.get('version') == 1 and isinstance(queue.get('items'), list)
def cover(source):
    suffix = source.suffix.lower()
    if suffix == '.pdf':
        with pdfium.PdfDocument(str(source)) as pdf:
            return pdf[0].render(scale=2).to_pil().convert('RGB')
    if suffix == '.epub':
        import xml.etree.ElementTree as ET
        from pathlib import PurePosixPath
        with zipfile.ZipFile(source) as book:
            container = ET.fromstring(book.read('META-INF/container.xml'))
            opf_path = container.find('.//{*}rootfile').attrib['full-path']
            opf = ET.fromstring(book.read(opf_path))
            wanted = next((m.attrib.get('content') for m in opf.findall('.//{*}meta') if m.attrib.get('name') == 'cover'), None)
            item = next((i for i in opf.findall('.//{*}item') if i.attrib.get('id') == wanted or 'cover-image' in i.attrib.get('properties', '')), None)
            if item is not None:
                cover_path = str(PurePosixPath(opf_path).parent / item.attrib['href'])
                return Image.open(io.BytesIO(book.read(cover_path))).convert('RGB')
    try:
        return Image.open(source).convert('RGB')
    except Exception:
        image = Image.new('RGB', (600, 800), '#171d2b')
        draw = ImageDraw.Draw(image)
        draw.text((40, 80), source.suffix.upper()[:12], fill='#90d5b8')
        draw.text((40, 180), source.stem[:50], fill='white')
        return image
def source_for(entry):
    parts = entry['path']
    if not parts or any(part in ('', '.', '..') or '/' in part or '\\' in part for part in parts):
        raise ValueError('Ruta inválida')
    path = root.joinpath(*parts)
    if entry.get('folder'):
        children = sorted((p for p in path.iterdir() if p.is_file()), key=lambda p:p.name.casefold())
        for extensions in ({'.jpg','.jpeg','.png','.webp'}, {'.pdf','.epub'}):
            match = next((p for p in children if p.suffix.lower() in extensions), None)
            if match: return match
        raise FileNotFoundError('Sin portada o PDF/EPUB en la raíz del contenedor')
    return path
out = hidden / 'portadas_lote.zip'
temp = hidden / 'portadas_lote.tmp'
failed = []
with zipfile.ZipFile(temp, 'w', compression=zipfile.ZIP_DEFLATED) as batch:
    for entry in queue['items']:
        ident = entry.get('drive_id', '')
        if not re.fullmatch(r'[A-Za-z0-9_-]{10,200}', ident):
            failed.append((ident, 'ID inválido')); continue
        try:
            image = ImageOps.contain(cover(source_for(entry)), (600, 800))
            canvas = Image.new('RGB', (600, 800), '#111722')
            canvas.paste(image, ((600-image.width)//2, (800-image.height)//2))
            data = io.BytesIO(); canvas.save(data, 'JPEG', quality=84, optimize=True)
            batch.writestr(ident + '.jpg', data.getvalue())
        except Exception as error:
            failed.append((ident, str(error)))
os.replace(temp, out)
print(f'Listo: {len(queue["items"])-len(failed)} portadas. Balisco importará el lote cuando Drive lo sincronice.')
if failed: print('Sin portada:', failed)
